# Phase 5 / 5b — LangGraph agentic layer + conversational contextualization

Companion exploration notebook for ROADMAP.md §Phase 5 and §Phase 5b. All the actual logic lives
in importable `rag/*.py` modules (`agent_state.py`, `planner.py`, `router.py`, `contextualize.py`,
`graph.py`, `generation.py`) — this notebook imports them and runs each piece in isolation so the
pipeline is inspectable cell by cell, rather than re-implementing anything here. `rag/agent_cli.py`
is the same pipeline as a one-shot CLI (or a multi-turn REPL via `--chat`); this notebook is for
understanding it, not a second copy.

Walk order:
1. State shape (`AgentState`, `BranchResult`)
2. Planner alone (decompose vs. bypass)
3. Router alone (category selection, fail-open)
4. Full graph — parallel fanout (ablation row 5)
5. Same question, `use_fanout=False` (ablation row 10 — isolates fanout as the only variable)
6. Checkpointing — replay a run's state after the fact
7. Conversational contextualization (Phase 5b) — follow-up rewriting, multi-turn end-to-end

**Needs a running Qdrant index** (`ai`/`cs.CL`/`finance`, per PHASE3_NOTES.md) and
`GOOGLE_API_KEY` in `.env`. Embedded-mode Qdrant is single-writer — close `retriever` (last
cell) before running `rag.cli`/`rag.agent_cli` from a separate process, or you'll hit
`StorageLockedError`.

In [1]:
# Unlike the Phase 1 notebooks (which predate rag/ and are fully self-contained), this one
# imports from the rag package - Jupyter's cwd is this notebook's own directory, not the repo
# root, so `from rag import ...` needs the root on sys.path first.
import sys
from pathlib import Path

_root = Path.cwd()
while not (_root / "rag" / "__init__.py").exists():
    if _root.parent == _root:
        raise RuntimeError("could not find the project root (looked for rag/__init__.py)")
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [9]:
from rag.generation import Generator
from rag.graph import build_graph, initial_state, run_agent
from rag.planner import planner_node
from rag.retrieval import Retriever
from rag.router import router_node

In [10]:
# Constructed once, reused for the whole notebook - matches rag/eval.py's run_tier23 style.
# Model/index loads happen lazily on first use (rag/retrieval.py, rag/generation.py), so this
# cell itself is instant; the first retrieval/generation call below is the slow one.
retriever = Retriever()
generator = Generator()

## 1. State shape

`AgentState` (rag/agent_state.py) is the graph's TypedDict schema. `branch_results` is the one
field with a non-default reducer (`operator.add`) - every parallel `topic_subagent` branch
returns one `BranchResult`, and LangGraph concatenates them onto this list automatically.

In [4]:
state = initial_state("How do transformer architectures work?")
state

{'question': 'How do transformer architectures work?',
 'history': [],
 'use_planner': True,
 'use_fanout': True,
 'use_context': True,
 'subtasks': [],
 'route': [],
 'branch_results': [],
 'category': None,
 'final_answer': None}

## 2. Planner alone

ROADMAP: "this is your query rewriting... make it bypassable so it becomes an ablation arm."
`planner_node(state, generator)` is a plain function - no graph needed to call it directly.

In [ ]:
# A genuinely compound question - expect more than one subtask back.
compound_state = {
    "question": "How do transformer architectures work",
    "use_planner": False,
    
}
planner_node(compound_state, generator)

{'subtasks': ['How do transformer architectures work?',
  'How are transformer architectures used in financial risk modeling?']}

In [6]:
# An atomic question - expect exactly one subtask, unchanged.
atomic_state = {"question": "What is the Black-Scholes model?", "use_planner": True}
planner_node(atomic_state, generator)

{'subtasks': ['What is the Black-Scholes model?']}

In [7]:
# use_planner=False: no LLM call at all, subtasks=[question] - the ablation-row-3 arm.
bypass_state = {"question": "What is the Black-Scholes model?", "use_planner": False}
planner_node(bypass_state, generator)

{'subtasks': ['What is the Black-Scholes model?']}

## 3. Router alone

`router_node(state, generator)` picks a subset of `config.INDEXED_CATEGORIES`. It fails open to
every indexed category on an empty/invalid LLM response (rag/router.py) - it should never route
to zero categories, since that would fan out to zero branches and produce no answer at all.

Try a single-topic question and a genuinely cross-topic one - the router should narrow to one
category for the first and pick more than one for the second (see `rag/router_eval.py` for a
scored version of this against goldset ground truth).

In [13]:
router_node({"question": "What is the Black-Scholes model used for in options pricing?"}, generator)

{'route': ['finance']}

In [14]:
router_node(
    {"question": "what is the drug in paracetomol?"}, generator
)

{'route': ['ai', 'cs.CL', 'finance']}

## 4. Full graph — parallel fanout (row 5)

`build_graph()` wires planner → router → (`Send` fanout | single_agent) → synthesizer, checkpointed
via SqliteSaver (`data/langgraph_checkpoints.sqlite`). `run_agent(..., use_fanout=True)` (the
default) fans out one parallel `topic_subagent` branch per routed category, then synthesizes them.

In [10]:
import uuid

question = "How do transformer architectures affect risk modeling in finance?"

# Fresh thread_id per execution, not a fixed string - re-running this notebook against an
# already-populated data/langgraph_checkpoints.sqlite with a REUSED thread_id resumes the prior
# checkpoint and accumulates onto branch_results (its operator.add reducer) instead of starting
# clean. Caught exactly this running the notebook a second time without wiping the checkpoint db
# first: branches came back as 6, not 3. Same class of bug documented in PHASE5B_NOTES.md §2,
# just found here instead of in the --chat REPL.
fanout_thread_id = f"notebook-fanout-demo-{uuid.uuid4()}"

graph = build_graph(retriever, generator)
fanout_result = run_agent(
    question, retriever, generator, use_fanout=True, thread_id=fanout_thread_id, graph=graph
)
print("route:", fanout_result["route"])
print("subtasks:", fanout_result["subtasks"])
print("branches:", len(fanout_result["branch_results"]))


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 26848.27it/s]


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 7209.77it/s]

route: ['finance', 'ai', 'cs.CL']
subtasks: ['How do transformer architectures affect risk modeling in finance?']
branches: 3


In [11]:
for branch in fanout_result["branch_results"]:
    print(f"[{branch.category}] {len(branch.chunks)} chunk(s), abstained={branch.draft.abstained}")
    print("  draft:", " ".join(branch.draft.text.split())[:200])
    print()

[finance] 5 chunk(s), abstained=True
  draft: INSUFFICIENT_CONTEXT The provided sources discuss the use of transformer-based models for measuring financial news sentiment and constructing mood indices, but they do not contain information regardin

[ai] 5 chunk(s), abstained=True
  draft: INSUFFICIENT_CONTEXT The provided sources discuss the use of transformer architectures for tracing Seiberg dualities in theoretical physics and do not contain information regarding risk modeling in fi

[cs.CL] 5 chunk(s), abstained=True
  draft: INSUFFICIENT_CONTEXT The provided sources discuss transformer architectures in the context of mental health detection and language modeling, but they do not contain information regarding risk modeling



In [12]:
answer = fanout_result["final_answer"]
print(answer.text)
print()
print("cited:", answer.cited, "| uncited:", answer.uncited, "| invalid:", answer.invalid)
print("usage:", answer.usage, "| latency_s:", round(answer.latency_s, 2))

INSUFFICIENT_CONTEXT
The provided sources discuss transformer architectures in the contexts of financial news sentiment analysis, mental health detection, and theoretical physics, but they do not contain information regarding the impact of these architectures on risk modeling in finance.

cited: [] | uncited: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15] | invalid: []
usage: {'input_tokens': 4844, 'output_tokens': 48, 'total_tokens': 4892, 'input_token_details': {'cache_read': 0}} | latency_s: 1.6


## 5. Row 10 — same scope, no fanout

Same question, `use_fanout=False`. Design decision (see PHASE5_NOTES.md): row 10 keeps the
planner's subtasks and the router's category scope - only the parallel-branch structure is
removed, replaced by one pooled retrieval + one generation call. That isolates fanout as the
only variable between this run and the one above, which is what ROADMAP §5's "how do you know
the parallel agents help? (row 5 vs row 10)" needs to be a clean comparison.

In [13]:
# single_agent_node never touches branch_results, so a fixed thread_id here can't accumulate
# stale branches the way fanout_thread_id could - but a fresh id per execution is the safer
# default regardless (matches rag/agent_cli.py's own uuid4() default for a single-shot run).
baseline_thread_id = f"notebook-baseline-demo-{uuid.uuid4()}"

baseline_result = run_agent(
    question, retriever, generator, use_fanout=False, thread_id=baseline_thread_id, graph=graph
)
print("route:", baseline_result["route"], "(compare to the fanout run's route above)")
print("branches:", len(baseline_result["branch_results"]), "(expect 0 - single_agent doesn't fan out)")
print()
print(baseline_result["final_answer"].text)

route: ['finance', 'ai', 'cs.CL'] (compare to the fanout run's route above)
branches: 0 (expect 0 - single_agent doesn't fan out)

INSUFFICIENT_CONTEXT
The provided sources discuss the use of transformer-based models for measuring financial news sentiment and forecasting macroeconomic expectations, but they do not contain information regarding how these architectures affect risk modeling.


In [14]:
fa, ba = fanout_result["final_answer"], baseline_result["final_answer"]
print(f"{'':12} {'fanout (row 5)':>18} {'baseline (row 10)':>20}")
print(f"{'in tokens':12} {fa.usage.get('input_tokens', '?'):>18} {ba.usage.get('input_tokens', '?'):>20}")
print(f"{'out tokens':12} {fa.usage.get('output_tokens', '?'):>18} {ba.usage.get('output_tokens', '?'):>20}")
print(f"{'latency_s':12} {fa.latency_s:>18.2f} {ba.latency_s:>20.2f}")
print(f"{'abstained':12} {str(fa.abstained):>18} {str(ba.abstained):>20}")

                 fanout (row 5)    baseline (row 10)
in tokens                  4844                 1338
out tokens                   48                   42
latency_s                  1.60                 1.14
abstained                  True                 True


## 6. Checkpointing — replay

Every `run_agent()` call above was checkpointed under its `thread_id`. `graph.get_state(...)`
reads that state back - the same mechanism works from a completely separate process reading
`data/langgraph_checkpoints.sqlite` directly (see PHASE5_NOTES.md's verification section).

In [15]:
snapshot = graph.get_state({"configurable": {"thread_id": fanout_thread_id}})
print("replayed question:", snapshot.values["question"])
print("replayed route:", snapshot.values["route"])
print("matches the live fanout_result:", snapshot.values["route"] == fanout_result["route"])

replayed question: How do transformer architectures affect risk modeling in finance?
replayed route: ['finance', 'ai', 'cs.CL']
matches the live fanout_result: True


## 7. Conversational contextualization (Phase 5b)

`contextualize_node` runs BEFORE the planner (`START -> contextualize -> planner -> ...`) and
rewrites a follow-up into a standalone question using the last `config.MAX_HISTORY_TURNS` (3,
per ROADMAP §7's explicit cut list) turns of history. Two free bypasses before any LLM call:
`use_context=False`, or no history yet (turn 1 of any conversation has nothing to resolve
against). History is caller-managed, not accumulated by the graph/checkpointer - see
PHASE5B_NOTES.md for why (`branch_results`' `operator.add` reducer would otherwise leak stale
branches across turns if the same `thread_id` were reused).

In [16]:
from rag.agent_state import Turn
from rag.contextualize import contextualize_node

# Turn 1 of any conversation: no history -> free bypass, no LLM call, question unchanged.
contextualize_node({"question": "what about the second one?", "history": [], "use_context": True}, generator)

{}

In [17]:
# With a prior turn in history, a genuinely ambiguous follow-up gets rewritten standalone.
one_turn_history = [Turn(
    question="What is the Black-Scholes model used for in options pricing?",
    answer="The provided sources do not contain information regarding the use of the "
           "Black-Scholes model in options pricing.",
)]
contextualize_node(
    {"question": "what about its assumptions?", "history": one_turn_history, "use_context": True},
    generator,
)

{'question': 'What are the assumptions of the Black-Scholes model?'}

Now end-to-end through the full graph: two turns, threading a `Turn` between them exactly like
`rag/agent_cli.py`'s `--chat` REPL does. **Each turn uses its own `thread_id`** - reusing one
shared `thread_id` across turns was tried first and found to leak the previous turn's
`branch_results` into the next one via its `operator.add` reducer (verified directly: a second
`graph.invoke()` on the same thread_id came back with the first turn's branch mixed in). Turn ids
below share a session prefix purely so they're identifiable as one conversation, not because they
share checkpoint state.

In [18]:
session_id = f"notebook-chat-demo-{uuid.uuid4()}"
chat_history: list[Turn] = []

turn1 = run_agent(
    "What is the Black-Scholes model used for in options pricing?", retriever, generator,
    use_fanout=False, history=chat_history, thread_id=f"{session_id}-turn-1", graph=graph,
)
print("turn 1 question:", turn1["question"])
print("turn 1 route:", turn1["route"])
chat_history.append(Turn(question=turn1["question"], answer=turn1["final_answer"].text))

turn 1 question: What is the Black-Scholes model used for in options pricing?
turn 1 route: ['finance']


In [19]:
turn2 = run_agent(
    "What about its assumptions?", retriever, generator,
    use_fanout=False, history=chat_history, thread_id=f"{session_id}-turn-2", graph=graph,
)
print("turn 2 raw follow-up: What about its assumptions?")
print("turn 2 resolved question:", turn2["question"])
print("turn 2 route:", turn2["route"])
print()
print(turn2["final_answer"].text)

turn 2 raw follow-up: What about its assumptions?
turn 2 resolved question: What are the assumptions of the Black-Scholes model in options pricing?
turn 2 route: ['finance']

INSUFFICIENT_CONTEXT
The provided sources do not contain information regarding the assumptions of the Black-Scholes model.


---

For the one-shot CLI form of everything above, see `rag/agent_cli.py`
(`python -m rag.agent_cli "..." [--no-planner] [--no-fanout] [--show-context]`), or
`python -m rag.agent_cli --chat` for the multi-turn REPL (Phase 5b). For the router's scored
evaluation against goldset ground truth, see `rag/router_eval.py`. Design decisions and defense
notes are in `PHASE5_NOTES.md` and `PHASE5B_NOTES.md`.

Run the next cell before starting `rag.cli`/`rag.agent_cli` from a terminal - embedded-mode
Qdrant is single-writer, and this notebook's `retriever` is still holding the storage lock.

In [20]:
retriever.close()